In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import joblib
import glob

# 1. Load labels + the raw sliced telemetry (need raw for feature stats)
labels = pd.read_parquet('../fastf1_data/labeled/labels_combined.parquet')
segmented_files = glob.glob('../fastf1_data/processed/corners_*.parquet')
segmented = pd.concat([pd.read_parquet(f) for f in segmented_files], ignore_index=True)

# 2. Build flat features — ONLY raw, unsmoothed columns (leakage prevention:
#    Throttle_smooth/Brake_smooth/Speed_smooth were used to CALCULATE the
#    labels, so they cannot be used as model inputs)
SAFE_COLUMNS = ['Throttle', 'Brake', 'Speed', 'nGear', 'RPM']

feature_rows = []
for (driver, lap_num, corner_num), group in segmented.groupby(['driver', 'lap_number', 'corner_number']):
    row = {'driver': driver, 'lap_number': lap_num, 'corner_number': corner_num}
    for col in SAFE_COLUMNS:
        row[f'{col}_mean'] = group[col].mean()
        row[f'{col}_std'] = group[col].std()
        row[f'{col}_min'] = group[col].min()
        row[f'{col}_max'] = group[col].max()
    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows)

# 3. Merge with labels (only keep rows that survived the out-lap filter in labeling)
data = features_df.merge(
    labels[['driver', 'lap_number', 'corner_number',
            'aggression_score', 'line_shape_score', 'oversteer_preference_score']],
    on=['driver', 'lap_number', 'corner_number']
)
data = data.dropna()
print(f"Final dataset: {data.shape}")

# 4. Split by DRIVER (not randomly) — prevents the model from
#    seeing the same driver's data in both train and validation
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(splitter.split(data, groups=data['driver']))
train_df, val_df = data.iloc[train_idx], data.iloc[val_idx]

feature_cols = [c for c in data.columns if c.endswith(('_mean', '_std', '_min', '_max'))]
target_cols = ['aggression_score', 'line_shape_score', 'oversteer_preference_score']

# 5. Scale (fit on train only)
scaler = StandardScaler()
train_df = train_df.copy()
val_df = val_df.copy()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
val_df[feature_cols] = scaler.transform(val_df[feature_cols])

# 6. Save everything models will need
train_df.to_parquet('../fastf1_data/labeled/train_data.parquet')
val_df.to_parquet('../fastf1_data/labeled/val_data.parquet')
joblib.dump(scaler, '../models/scaler.pkl')

print(f"Train: {train_df.shape}, Val: {val_df.shape}")
print(f"Feature columns: {feature_cols}")

Final dataset: (42573, 26)


FileNotFoundError: [Errno 2] No such file or directory: '../models/scaler.pkl'